<div dir="rtl">

# الدرس 46: الانتباه الذاتي باليد

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الانتباه بنمباي

</div>

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def attention(Q, K, V, causal=False):
    scores = Q @ K.T / np.sqrt(K.shape[-1])            # (n, n)
    if causal:
        future = np.triu(np.ones_like(scores, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)      # hide the future
    weights = softmax(scores)                           # each row sums to 1
    return weights @ V, weights

Q = np.array([[1, 0, 1, 0], [0, 2, 0, 2], [2, 0, 2, 0]], dtype=float)
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 1, 0, 0]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)

out, w = attention(Q, K, V)
print("weights:\n", w.round(3), "\noutput:\n", out.round(3))
out_c, w_c = attention(Q, K, V, causal=True)
print("causal weights:\n", w_c.round(3), "\ncausal output:\n", out_c.round(3))

<div dir="rtl">

### نفس الحساب بدالة باي تورش الجاهزة

</div>

In [ ]:
Qt, Kt, Vt = (torch.tensor(m, dtype=torch.float32) for m in (Q, K, V))
print(F.scaled_dot_product_attention(Qt, Kt, Vt).numpy().round(3))
print(F.scaled_dot_product_attention(Qt, Kt, Vt, is_causal=True).numpy().round(3))

<div dir="rtl">

### طبقة انتباه كاملة

</div>

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)

    def forward(self, x):                                   # x: (batch, n, d_model)
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)     # (batch, n, d_k)
        scores = q @ k.transpose(-2, -1) / k.shape[-1] ** 0.5    # (batch, n, n)
        n = x.shape[1]
        future = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float("-inf"))
        weights = scores.softmax(dim=-1)
        return weights @ v, weights

torch.manual_seed(0)
layer = SelfAttention(d_model=8, d_k=8)
x = torch.randn(2, 5, 8)                     # 2 sentences, 5 tokens, 8 numbers each
y, weights = layer(x)
print("output:", tuple(y.shape), " weights:", tuple(weights.shape))
print("first sentence weights:\n", weights[0].detach().numpy().round(2))
print("parameters:", sum(p.numel() for p in layer.parameters()))

<div dir="rtl">

### أثر القسمة على الجذر

</div>

In [ ]:
rng = np.random.default_rng(0)
for d in [4, 64, 512]:
    q = rng.normal(size=(1000, d))
    k = rng.normal(size=(1000, 10, d))                   # 10 keys per query
    raw = np.einsum("nd,nkd->nk", q, k)
    top_raw = softmax(raw).max(axis=1).mean()
    top_scaled = softmax(raw / np.sqrt(d)).max(axis=1).mean()
    print(f"d = {d:3}: score std {raw.std():5.1f} | top weight without scaling {top_raw:.2f}, "
          f"with scaling {top_scaled:.2f}")

<div dir="rtl">

## تمرين: الرمز الأول، بدون قناع

استخدم مفاتيح وقيم المثال المحلول. استعلام الرمز الأول: $q = (1, 0, 1, 0)$.

1. احسب بيدك الدرجات، ثم القسمة على الجذر، ثم الأوزان، ثم المخرج، بدون قناع.
2. ماذا يصبح مخرج الرمز الأول مع القناع السببي؟ ولماذا لا تحتاج أي حساب؟
3. تحقق من الصف الأول في ناتج الكود.

</div>

In [ ]:
# اكتب حلّك هنا
